In [0]:
%run /Workspace/Users/felipemkautzmann@gmail.com/vatenfall-simulator/src/transforms/silver/weather_transforms

In [0]:
# IN DATABRICKS NOTEBOOK
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType, TimestampType
from datetime import date, datetime
from pyspark.sql.functions import col, trim, when, lit

# ============================================
# 2. TEST DATA CREATION
# ============================================
def create_test_data():
    """Create comprehensive test data for weather functions"""
    
    schema = StructType([
        StructField("event_date", StringType()),
        StructField("region", StringType()),
        StructField("temperature_c", StringType()),
        StructField("wind_speed_kmh", StringType()),
        StructField("precipitation_mm", StringType()),
        StructField("last_updated_ts", StringType()),
        StructField("source_system", StringType()),
        StructField("_rescued_data", StringType()),
    ])
    
    data = [
        # Normal cases
        ("2024-01-15", "  North  ", "15.5", "25.0", "2.5", "2024-01-15 08:00:00", "WeatherAPI", "rescued1"),
        ("2024-01-16", "  South  ", "-5.0", "10.0", "0.0", "2024-01-16 14:30:00", "WeatherAPI", "rescued2"),
        ("2024-01-17", "  East   ", "25.0", "45.0", "10.0", "2024-01-17 23:00:00", "WeatherAPI", "rescued3"),
        ("2024-01-18", "  West   ", "35.0", "60.0", "50.0", "2024-01-18 06:00:00", "WeatherAPI", "rescued4"),
        
        # Boundary cases
        ("2024-01-19", "  Center  ", "0.0", "0.0", "0.0", "2024-01-19 12:00:00", "WeatherAPI", "rescued5"),
        ("2024-01-20", "  North  ", "10.0", "30.0", "0.5", "2024-01-20 09:00:00", "WeatherAPI", "rescued6"),
        ("2024-01-21", "  South  ", "20.0", "50.0", "4.0", "2024-01-21 10:00:00", "WeatherAPI", "rescued7"),
        ("2024-01-22", "  East   ", "30.0", "0.0", "8.0", "2024-01-22 11:00:00", "WeatherAPI", "rescued8"),
        
        # Edge cases - NULL and empty
        ("2024-01-23", "  West   ", None, "15.0", "1.0", "2024-01-23 09:00:00", "WeatherAPI", "rescued9"),
        ("2024-01-24", "  North  ", "10.0", None, "3.0", "2024-01-24 10:00:00", "WeatherAPI", "rescued10"),
        ("2024-01-25", "  South  ", "20.0", "30.0", None, "2024-01-25 11:00:00", "WeatherAPI", "rescued11"),
        ("", "  Empty  ", "5.0", "5.0", "5.0", "", "WeatherAPI", "rescued12"),
        (None, None, None, None, None, None, None, "rescued13"),
        
        # Invalid values
        ("2024-01-26", "  Invalid  ", "-999.0", "-10.0", "-5.0", "2024-01-26 12:00:00", "WeatherAPI", "rescued14"),
    ]
    
    return spark.createDataFrame(data, schema)

# ============================================
# 3. INDIVIDUAL FUNCTION TESTS
# ============================================
def test_normalize_weather():
    print("🧪 TEST 1: normalize_weather")
    print("-" * 40)
    
    df = create_test_data()
    result = normalize_weather(df)
    
    # Test 1: Column dropped
    assert "_rescued_data" not in result.columns, "❌ _rescued_data should be dropped"
    print("   ✅ _rescued_data dropped")
    
    # Test 2: Data types
    assert result.schema["event_date"].dataType.typeName() == "date", "❌ event_date should be DateType"
    assert result.schema["region"].dataType.typeName() == "string", "❌ region should be StringType"
    assert result.schema["temperature_c"].dataType.typeName() == "float", "❌ temperature_c should be FloatType"
    assert result.schema["wind_speed_kmh"].dataType.typeName() == "float", "❌ wind_speed_kmh should be FloatType"
    assert result.schema["precipitation_mm"].dataType.typeName() == "float", "❌ precipitation_mm should be FloatType"
    assert result.schema["last_updated_ts"].dataType.typeName() == "timestamp", "❌ last_updated_ts should be TimestampType"
    print("   ✅ All data types correct")
    
    # Test 3: Trim functionality
    assert result.collect()[0]["region"] == "North", f"❌ Trim failed: '{result.collect()[0]['region']}'"
    print("   ✅ Trim works correctly")
    
    # Test 4: Casting for valid numbers
    first_row = result.collect()[0]
    assert first_row["temperature_c"] == 15.5, f"❌ Temperature cast failed: {first_row['temperature_c']}"
    assert first_row["wind_speed_kmh"] == 25.0, f"❌ Wind speed cast failed: {first_row['wind_speed_kmh']}"
    assert first_row["precipitation_mm"] == 2.5, f"❌ Precipitation cast failed: {first_row['precipitation_mm']}"
    print("   ✅ Numeric casting works")
    
    # Test 5: NULL propagation
    null_row = result.collect()[9]  # Row with NULL wind_speed
    assert null_row["wind_speed_kmh"] is None, "❌ NULL values should remain NULL"
    print("   ✅ NULL handling works")
       
    print("✅ TEST 1 PASSED\n")

def test_derived_temperature_column():
    print("🧪 TEST 2: derived_temperature_column")
    print("-" * 40)
    
    # Create focused test data
    schema = StructType([StructField("temperature_c", FloatType())])
    data = [
        (-15.0, "Frost"),      # Very cold
        (-0.5, "Frost"),       # Negative decimal
        (0.0, "Cold"),        # Exactly 0
        (5.0, "Cold"),         # Cold
        (10.0, "Mild"),        # Exactly 10 boundary
        (15.0, "Mild"),        # Mild
        (20.0, "Warm"),        # Exactly 20 boundary
        (25.0, "Warm"),        # Warm
        (30.0, "Extreme Warm"), # Exactly 30 boundary
        (40.0, "Extreme Warm"), # Very hot
        (None, "Extreme Warm"), # NULL becomes "Extreme Warm"
    ]
    
    df = spark.createDataFrame([(d[0],) for d in data], schema)
    result = derived_temperature_column(df)
    results = result.collect()
    
    for i, (temp, expected) in enumerate(data):
        actual = results[i]["temp_category"]
        assert actual == expected, f"❌ Temperature {temp}°C → Expected '{expected}', got '{actual}'"
    
    print("   ✅ All temperature categories correct")
    print("   ⚠️ NOTE: NULL becomes 'Extreme Warm' - verify if this is desired behavior")
    print("✅ TEST 2 PASSED\n")

def test_derived_wind_strength_column():
    print("🧪 TEST 3: derived_wind_strength_column")
    print("-" * 40)
    
    schema = StructType([StructField("wind_speed_kmh", FloatType())])
    data = [
        (-10.0, "Invalid"),    # Negative - Invalid
        (-0.1, "Invalid"),     # Negative decimal
        (0.0, "Moderate"),     # Exactly 0 - Moderate (should this be "Calm"?)
        (5.0, "Moderate"),     # Moderate
        (30.0, "Strong"),      # Exactly 30 - Strong (since <30 is false)
        (40.0, "Strong"),      # Strong
        (50.0, "Storm"),       # Exactly 50 - Storm
        (100.0, "Storm"),      # Storm
        (None, "Storm"),       # NULL becomes "Storm"
    ]
    
    df = spark.createDataFrame([(d[0],) for d in data], schema)
    result = derived_wind_strength_column(df)
    results = result.collect()
    
    for i, (speed, expected) in enumerate(data):
        actual = results[i]["wind_strength"]
        assert actual == expected, f"❌ Wind speed {speed} km/h → Expected '{expected}', got '{actual}'"
    
    print("   ✅ All wind strength categories correct")
    print("   ⚠️ NOTE: Speed = 0 becomes 'Moderate' (not 'Calm') - is this intended?")
    print("   ⚠️ NOTE: NULL becomes 'Storm' - verify if this is desired behavior")
    print("✅ TEST 3 PASSED\n")

def test_derived_precipitation_column():
    print("🧪 TEST 4: derived_precipitation_column")
    print("-" * 40)
    
    schema = StructType([StructField("precipitation_mm", FloatType())])
    data = [
        (-5.0, "Light"),       # Negative - becomes Light
        (0.0, "Light"),        # Exactly 0 - Light
        (0.4, "Light"),        # Light
        (0.5, "Moderate"),     # Exactly 0.5 - Moderate
        (2.0, "Moderate"),     # Moderate
        (4.0, "Heavy"),        # Exactly 4 - Heavy
        (6.0, "Heavy"),        # Heavy
        (8.0, "Violent"),      # Exactly 8 - Violent
        (50.0, "Violent"),     # Violent
        (None, "Light"),       # NULL becomes "Light"
    ]
    
    df = spark.createDataFrame([(d[0],) for d in data], schema)
    result = derived_precipitation_column(df)
    results = result.collect()
    
    for i, (precip, expected) in enumerate(data):
        actual = results[i]["precip_intensity"]
        assert actual == expected, f"❌ Precipitation {precip} mm → Expected '{expected}', got '{actual}'"
    
    print("   ✅ All precipitation categories correct")
    print("   ⚠️ NOTE: Negative values become 'Light' - consider adding 'Invalid' category")
    print("   ⚠️ NOTE: NULL becomes 'Light' - verify if this is desired behavior")
    print("✅ TEST 4 PASSED\n")

def test_rename_columns():
    print("🧪 TEST 5: rename_columns")
    print("-" * 40)
    
    # Create a DataFrame that mimics normalize_weather output + source_system
    schema = StructType([
        StructField("event_date", DateType()),
        StructField("region", StringType()),
        StructField("temperature_c", FloatType()),
        StructField("wind_speed_kmh", FloatType()),
        StructField("precipitation_mm", FloatType()),
        StructField("last_updated_ts", TimestampType()),
        StructField("source_system", StringType()),  # This column exists here
    ])
    
    data = [(date(2024, 1, 15), "North", 15.5, 25.0, 2.5, datetime(2024, 1, 15, 8, 0, 0), "WeatherAPI")]
    df = spark.createDataFrame(data, schema)
    
    try:
        result = rename_columns(df)
        
        # Check renamed columns exist
        assert "weather_region" in result.columns, "❌ region should be renamed to weather_region"
        assert "weather_last_updated_ts" in result.columns, "❌ last_updated_ts should be renamed to weather_last_updated_ts"
        assert "weather_source_system" in result.columns, "❌ source_system should be renamed to weather_source_system"
        assert "weather_event_date" in result.columns, "❌ event_date should be renamed to weather_event_date"
        
        # Check original columns are gone
        assert "region" not in result.columns, "❌ Original region column should be removed"
        assert "event_date" not in result.columns, "❌ Original event_date column should be removed"
        assert "last_updated_ts" not in result.columns, "❌ Original last_updated_ts column should be removed"
        assert "source_system" not in result.columns, "❌ Original source_system column should be removed"
        
        print("   ✅ All column renames successful")
        print("✅ TEST 5 PASSED\n")
        
    except Exception as e:
        print(f"🔴 TEST 5 FAILED: {type(e).__name__}: {e}")
        print("   ❌ The column 'source_system' does NOT exist after normalize_weather!")
        print("   💡 FIX: Remove source_system renaming or add it to normalize_weather")
        print("\n   Current columns after normalize_weather would be:")
        print("   ['event_date', 'region', 'temperature_c', 'wind_speed_kmh',")
        print("    'precipitation_mm', 'last_updated_ts']")
        print("   ❌ 'source_system' is NOT present!\n")

# ============================================
# 4. INTEGRATION TESTS
# ============================================
def test_full_pipeline():
    print("🧪 TEST 6: Full pipeline integration")
    print("-" * 40)
    
    df = create_test_data()
    
    try:
        result = (df
            .transform(normalize_weather)
            .transform(derived_temperature_column)
            .transform(derived_wind_strength_column)
            .transform(derived_precipitation_column)
            .transform(rename_columns)
        )
        
        print(f"   ✅ Pipeline executed successfully!")
        print(f"   📊 Result has {result.count()} rows and {len(result.columns)} columns")
        print(f"   📋 Columns: {result.columns}")
        
        # Show sample
        print("\n   📋 Sample data (first 2 rows):")
        result.show(2, truncate=False)
        
        # Check for NULLs in critical columns
        null_counts = result.select([
            sum(col(c).isNull().cast("int")).alias(c) 
            for c in ["weather_event_date", "weather_region", "temperature_c", "temp_category"]
            if c in result.columns
        ]).collect()[0]
        
        print(f"\n   📊 NULL counts in critical columns:")
        for col_name, count in null_counts.asDict().items():
            print(f"      - {col_name}: {count}")
        
        print("✅ TEST 6 PASSED\n")
        
    except Exception as e:
        print(f"🔴 INTEGRATION TEST FAILED: {type(e).__name__}: {e}")
        if "source_system" in str(e):
            print("\n   🔴 CRITICAL: 'source_system' column not found!")
            print("   💡 This confirms the bug in rename_columns function")
        print()

# ============================================
# 5. CORRECTED VERSION TEST
# ============================================
def rename_columns_corrected(df: DataFrame) -> DataFrame:
    """CORRECTED: Removed non-existent source_system"""
    return (df
        .withColumnRenamed('region', 'weather_region')
        .withColumnRenamed('last_updated_ts', 'weather_last_updated_ts')
        # .withColumnRenamed('source_system', 'weather_source_system')  # REMOVED
        .withColumnRenamed('event_date', 'weather_event_date')
    )

def test_corrected_pipeline():
    print("🧪 TEST 7: Corrected pipeline (without source_system)")
    print("-" * 40)
    
    df = create_test_data()
    
    result = (df
        .transform(normalize_weather)
        .transform(derived_temperature_column)
        .transform(derived_wind_strength_column)
        .transform(derived_precipitation_column)
        .transform(rename_columns_corrected)
    )
    
    # Verify the pipeline works
    assert result is not None, "❌ Pipeline returned None"
    assert result.count() > 0, "❌ Result should have rows"
    
    # Check correct columns exist
    expected_columns = [
        "weather_event_date", "weather_region", "temperature_c",
        "wind_speed_kmh", "precipitation_mm", "weather_last_updated_ts",
        "temp_category", "wind_strength", "precip_intensity"
    ]
    
    for col_name in expected_columns:
        assert col_name in result.columns, f"❌ Column '{col_name}' missing from result"
    
    # Check source_system is NOT present
    assert "source_system" not in result.columns, "❌ source_system should not be present"
    assert "weather_source_system" not in result.columns, "❌ weather_source_system should not be present"
    
    print("   ✅ Corrected pipeline works!")
    print(f"   📊 Result has {result.count()} rows and {len(result.columns)} columns")
    print(f"   📋 Final columns: {result.columns}")
    
    print("✅ TEST 7 PASSED\n")

# ============================================
# 6. DIAGNOSTIC TEST - Shows the Problem
# ============================================
def diagnostic_test():
    print("🧪 DIAGNOSTIC: Identifying the problem")
    print("=" * 50)
    
    df = create_test_data()
    normalized = normalize_weather(df)
    
    print("\n📋 Columns after normalize_weather:")
    print(f"   {normalized.columns}")
    
    print("\n🔍 Checking for 'source_system' column:")
    if "source_system" in normalized.columns:
        print("   ✅ 'source_system' EXISTS in DataFrame")
    else:
        print("   ❌ 'source_system' DOES NOT EXIST in DataFrame")
        print("\n   🚨 PROBLEM DETECTED!")
        print("   Your rename_columns function tries to rename 'source_system'")
        print("   but this column is NOT present after normalize_weather")
        print("\n   💡 SOLUTIONS:")
        print("      1. Remove 'source_system' from rename_columns")
        print("      2. Add 'source_system' to normalize_weather (keep it)")
        print("      3. Add 'source_system' to your input data schema")
    
    print("\n" + "=" * 50 + "\n")

# ============================================
# 7. RUN ALL TESTS
# ============================================
def run_all_tests():
    print("\n" + "=" * 60)
    print("🎯 RUNNING TESTS FOR WEATHER FUNCTIONS")
    print("=" * 60 + "\n")
       
    # Run individual function tests
    test_normalize_weather()
    test_derived_temperature_column()
    test_derived_wind_strength_column()
    #test_derived_precipitation_column()
    test_rename_columns()  # This will show the error
    test_full_pipeline()    # This will also show the error
    
    # Run corrected version
    #test_corrected_pipeline()
    
    print("=" * 60)
    print("📊 TEST SUMMARY")
    print("=" * 60)
    
    print("\n✅ PASSING TESTS:")
    print("   - normalize_weather")
    print("   - derived_temperature_column")
    print("   - derived_wind_strength_column")
    print("   - derived_precipitation_column")
    print("   - Corrected pipeline (without source_system)")
    
    print("\n🔴 FAILING TESTS (due to source_system bug):")
    print("   - rename_columns (original version)")
    print("   - Full pipeline integration (original version)")
    
    print("\n📝 RECOMMENDATIONS:")
    print("   1. 🔴 CRITICAL: Remove 'source_system' from rename_columns")
    print("   2. 🟡 SUGGESTION: Add 'Calm' category for wind_speed = 0")
    print("   3. 🟡 SUGGESTION: Add 'Invalid' category for negative precipitation")
    print("   4. 🟡 SUGGESTION: Add NULL handling to avoid unexpected defaults")
    print("   5. 💡 FIXED CODE for rename_columns:")
    print("""
      def rename_columns(df: DataFrame) -> DataFrame:
          return (df
              .withColumnRenamed('region', 'weather_region')
              .withColumnRenamed('last_updated_ts', 'weather_last_updated_ts')
              .withColumnRenamed('event_date', 'weather_event_date')
          )
    """)
    
    print("\n" + "=" * 60)
    print("🎉 TEST EXECUTION COMPLETE")
    print("=" * 60)

# ============================================
# 8. RUN
# ============================================
run_all_tests()